# JIRA Ticket Scorer with LangFlow

This notebook demonstrates how to:
1. Connect to JIRA API
2. Fetch ticket data
3. Use LangFlow to score tickets
4. Export results

In [1]:
# Install required packages
!pip install requests langflow openai jira pandas --quiet

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/Users/hari.durai/automationProject/jira-scorer_2/backend/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Users/hari.durai/automationProject/jira-scorer_2/backend/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/Users/hari.durai/automationProject/jira-scorer_2/backend/venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/private/var/folders/q

In [2]:
import requests
import json
from typing import Dict, List
import pandas as pd
from pathlib import Path

ModuleNotFoundError: No module named 'pandas'

## Configuration

In [ ]:
# Configuration
JIRA_URL = "https://your-domain.atlassian.net"  # Your JIRA instance URL
JIRA_USERNAME = "your-email@example.com"  # Your JIRA username (email)
JIRA_TOKEN = "your-jira-api-token"  # Your JIRA API token
OPENAI_API_KEY = "sk-proj-your-openai-api-key"  # Your OpenAI API key

# Load custom prompt
PROMPT_FILE = Path('scoring_prompt.md')

## Load Scoring Prompt

In [ ]:
def load_prompt(prompt_file: Path) -> str:
    """Load the scoring prompt from markdown file"""
    if prompt_file.exists():
        with open(prompt_file, 'r') as f:
            return f.read()
    else:
        print(f"Warning: {prompt_file} not found, using default prompt")
        return """Score this JIRA ticket based on:
1. Description clarity (0-10)
2. Technical details completeness (0-10)
3. Acceptance criteria quality (0-10)

Provide an overall score (0-10) and brief feedback."""

scoring_prompt = load_prompt(PROMPT_FILE)
print("Loaded prompt:")
print(scoring_prompt)

## JIRA API Functions

In [ ]:
def get_jira_ticket(jira_url: str, ticket_id: str, jira_username: str, auth_token: str) -> Dict:
    """Fetch a JIRA ticket via API"""
    # Create basic auth string
    import base64
    auth_string = f"{jira_username}:{auth_token}"
    auth_bytes = auth_string.encode('ascii')
    base64_auth = base64.b64encode(auth_bytes).decode('ascii')
    
    headers = {
        'Authorization': f'Basic {base64_auth}',
        'Content-Type': 'application/json'
    }
    
    url = f"{jira_url}/rest/api/3/issue/{ticket_id}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch JIRA ticket: {response.status_code} - {response.text}")

def extract_ticket_info(ticket_data: Dict) -> Dict:
    """Extract relevant information from JIRA ticket"""
    fields = ticket_data.get('fields', {})
    
    # Get description (handle different formats)
    description = fields.get('description', '')
    if isinstance(description, dict):
        # Handle Atlassian Document Format
        description = json.dumps(description, indent=2)
    
    return {
        'key': ticket_data.get('key', ''),
        'summary': fields.get('summary', ''),
        'description': description,
        'acceptance_criteria': fields.get('customfield_10000', ''),  # Adjust field ID as needed
        'labels': fields.get('labels', []),
        'priority': fields.get('priority', {}).get('name', '') if fields.get('priority') else '',
        'status': fields.get('status', {}).get('name', ''),
        'issue_type': fields.get('issuetype', {}).get('name', '')
    }

def search_jira_tickets(jira_url: str, jql: str, jira_username: str, auth_token: str, max_results: int = 50) -> List[str]:
    """Search for JIRA tickets using JQL"""
    import base64
    auth_string = f"{jira_username}:{auth_token}"
    auth_bytes = auth_string.encode('ascii')
    base64_auth = base64.b64encode(auth_bytes).decode('ascii')
    
    headers = {
        'Authorization': f'Basic {base64_auth}',
        'Content-Type': 'application/json'
    }
    
    url = f"{jira_url}/rest/api/3/search"
    params = {
        'jql': jql,
        'maxResults': max_results,
        'fields': 'key'
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        return [issue['key'] for issue in data.get('issues', [])]
    else:
        raise Exception(f"Failed to search JIRA: {response.status_code} - {response.text}")

## LangFlow / Claude Integration

In [ ]:
def score_ticket_with_claude(ticket_info: Dict, api_key: str, prompt_template: str) -> Dict:
    """Score ticket using OpenAI API"""
    
    # Build the full prompt
    ticket_text = f"""
JIRA Ticket: {ticket_info['key']}
Type: {ticket_info['issue_type']}
Summary: {ticket_info['summary']}

Description:
{ticket_info['description']}

Acceptance Criteria:
{ticket_info['acceptance_criteria']}

Priority: {ticket_info['priority']}
Status: {ticket_info['status']}
Labels: {', '.join(ticket_info['labels'])}
"""
    
    full_prompt = f"{prompt_template}\n\n{ticket_text}\n\nPlease provide the scoring in a structured format."
    
    # Call OpenAI API
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }
    
    payload = {
        'model': 'gpt-4o',
        'messages': [
            {
                'role': 'user',
                'content': full_prompt
            }
        ],
        'temperature': 0.7,
        'max_tokens': 1500
    }
    
    response = requests.post(
        'https://api.openai.com/v1/chat/completions',
        headers=headers,
        json=payload
    )
    
    if response.status_code == 200:
        result = response.json()
        return {
            'ticket_key': ticket_info['key'],
            'score_text': result['choices'][0]['message']['content'],
            'model': result['model'],
            'usage': result['usage']
        }
    else:
        raise Exception(f"Failed to score ticket: {response.status_code} - {response.text}")

## Example: Score a Single Ticket

In [ ]:
# Example: Score a single ticket
TICKET_ID = "PROJ-123"  # Replace with your ticket ID

try:
    # Fetch ticket
    ticket_data = get_jira_ticket(JIRA_URL, TICKET_ID, JIRA_USERNAME, JIRA_TOKEN)
    ticket_info = extract_ticket_info(ticket_data)
    
    print(f"Fetched ticket: {ticket_info['key']}")
    print(f"Summary: {ticket_info['summary']}")
    print("\n" + "="*50 + "\n")
    
    # Score ticket
    score_result = score_ticket_with_claude(ticket_info, OPENAI_API_KEY, scoring_prompt)
    
    print("Scoring Result:")
    print(score_result['score_text'])
    print(f"\nTokens used: {score_result['usage']}")
    
except Exception as e:
    print(f"Error: {e}")

## Batch Processing: Score Multiple Tickets

In [ ]:
# Example: Search and score multiple tickets
JQL_QUERY = "project = PROJ AND status = 'To Do' ORDER BY created DESC"  # Customize your JQL

try:
    # Search for tickets
    ticket_ids = search_jira_tickets(JIRA_URL, JQL_QUERY, JIRA_USERNAME, JIRA_TOKEN, max_results=10)
    print(f"Found {len(ticket_ids)} tickets: {', '.join(ticket_ids)}")
    
    # Score all tickets
    results = []
    
    for idx, ticket_id in enumerate(ticket_ids, 1):
        print(f"\nProcessing {idx}/{len(ticket_ids)}: {ticket_id}...")
        
        try:
            ticket_data = get_jira_ticket(JIRA_URL, ticket_id, JIRA_USERNAME, JIRA_TOKEN)
            ticket_info = extract_ticket_info(ticket_data)
            score_result = score_ticket_with_claude(ticket_info, OPENAI_API_KEY, scoring_prompt)
            
            results.append({
                'ticket_key': ticket_id,
                'summary': ticket_info['summary'],
                'status': ticket_info['status'],
                'priority': ticket_info['priority'],
                'score': score_result['score_text'],
                'success': True
            })
            
        except Exception as e:
            print(f"  Error scoring {ticket_id}: {e}")
            results.append({
                'ticket_key': ticket_id,
                'error': str(e),
                'success': False
            })
    
    # Create DataFrame for analysis
    df = pd.DataFrame(results)
    print("\n" + "="*50)
    print("Results Summary:")
    print(df[['ticket_key', 'summary', 'priority', 'success']].to_string())
    
except Exception as e:
    print(f"Error: {e}")

## Export Results

In [ ]:
# Export results to CSV
if 'df' in locals() and not df.empty:
    output_file = 'jira_ticket_scores.csv'
    df.to_csv(output_file, index=False)
    print(f"Results exported to {output_file}")
    
    # Export to JSON for more detailed data
    json_output = 'jira_ticket_scores.json'
    with open(json_output, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Detailed results exported to {json_output}")

## Visualization (Optional)

In [ ]:
# Optional: Visualize results
import matplotlib.pyplot as plt

if 'df' in locals() and not df.empty:
    # Example visualization - customize based on your scoring format
    successful_results = df[df['success'] == True]
    
    print(f"\nSuccessfully scored {len(successful_results)}/{len(df)} tickets")
    
    # You can add more visualizations here based on extracted scores